# CredResolve Collections Analytics — Independent Investigation

**Objective:** test the reported 11% MoM recovery improvement, reconstruct performance, identify drivers and data-quality failures, and recommend where ₹10 Cr should be invested.

**Important:** the supplied event data covers Jan 1–Aug 8, 2026; Jan–Jul are complete months and August is partial.

In [1]:
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
RAW=Path('../data/raw')
read=lambda n: pd.read_csv(RAW/f'{n}.csv')


## 1. Data quality first
The assignment explicitly warns that the data may contain duplicates, missing values, conflicting timestamps, inconsistent IDs and changed schemas. We therefore clean before calculating business KPIs.

In [2]:
borrowers=read('borrowers')
accounts=read('accounts')
payments_raw=read('payments')
calls_raw=read('calls')
print('borrowers:', borrowers.shape, 'exact duplicates:', borrowers.duplicated().sum())
print('accounts:', accounts.shape, 'missing borrower_id:', accounts.borrower_id.isna().sum())
print('payments:', payments_raw.shape, 'exact duplicates:', payments_raw.duplicated().sum())
print('calls:', calls_raw.shape, 'exact duplicates:', calls_raw.duplicated().sum())


borrowers: (30600, 8) exact duplicates: 600
accounts: (30000, 11) missing borrower_id: 455
payments: (25500, 9) exact duplicates: 486
calls: (91350, 11) exact duplicates: 1271


### Source-of-truth decisions
- `account_id` is the authoritative account key.
- `accounts.borrower_id` is the authoritative account→borrower relationship.
- Borrower profile uses the latest `updated_at` snapshot per borrower.
- Payment dedupe uses `payment_id`, not `payment_reference`, because references are reused across accounts.
- Timestamps are normalized to UTC when a row-level timezone exists.

In [3]:
# Payment cleaning
p=payments_raw.drop_duplicates().copy()
p['event_at']=pd.to_datetime(p.event_at,utc=True)
p['ref_present']=p.payment_reference.notna().astype(int)
p=p.sort_values(['payment_id','ref_present','event_at']).drop_duplicates('payment_id',keep='last').drop(columns='ref_present')
p['month']=p.event_at.dt.to_period('M').astype(str)
raw_success=payments_raw.query("payment_status=='SUCCESS'").amount.sum()
clean_success=p.query("payment_status=='SUCCESS'").amount.sum()
print(f'Exact-duplicate payment inflation: ₹{raw_success-clean_success:,.0f} ({(raw_success/clean_success-1)*100:.2f}%)')


Exact-duplicate payment inflation: ₹25,901,962 (1.97%)


C:\Users\LOQ\AppData\Local\Temp\ipykernel_9128\2550795992.py:6: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  p['month']=p.event_at.dt.to_period('M').astype(str)


## 2. Reconstruct monthly performance

In [4]:
target=read('daily_targeting'); target['target_date']=pd.to_datetime(target.target_date,utc=True); target['month']=target.target_date.dt.to_period('M').astype(str)
calls=calls_raw.drop_duplicates().sort_values('event_at').drop_duplicates('call_id',keep='last').copy(); calls['event_at']=pd.to_datetime(calls.event_at,utc=True); calls['month']=calls.event_at.dt.to_period('M').astype(str)
s=read('agent_sessions'); s['login_at']=pd.to_datetime(s.login_at,utc=True); s['logout_at']=pd.to_datetime(s.logout_at,utc=True); s['month']=s.login_at.dt.to_period('M').astype(str); s['hours']=(s.logout_at-s.login_at).dt.total_seconds()/3600
cd=calls[calls.direction=='OUTBOUND']
m=pd.DataFrame(index=sorted(target.month.unique()))
m['targeted_accounts']=target.groupby('month').account_id.nunique()
m['success_recovery']=p.query("payment_status=='SUCCESS'").groupby('month').amount.sum()
m['unique_payers']=p.query("payment_status=='SUCCESS'").groupby('month').account_id.nunique()
m['session_hours']=s.groupby('month').hours.sum()
m['recovery_per_target']=m.success_recovery/m.targeted_accounts
m['payer_rate']=m.unique_payers/m.targeted_accounts
m['recovery_per_agent_hour']=m.success_recovery/m.session_hours
m['answer_rate']=cd.groupby('month').apply(lambda x:(x.call_status=='ANSWERED').mean())
m=m.loc[['2026-01','2026-02','2026-03','2026-04','2026-05','2026-06','2026-07']]
m.round(4)


C:\Users\LOQ\AppData\Local\Temp\ipykernel_9128\18749289.py:1: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  target=read('daily_targeting'); target['target_date']=pd.to_datetime(target.target_date,utc=True); target['month']=target.target_date.dt.to_period('M').astype(str)
C:\Users\LOQ\AppData\Local\Temp\ipykernel_9128\18749289.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  calls=calls_raw.drop_duplicates().sort_values('event_at').drop_duplicates('call_id',keep='last').copy(); calls['event_at']=pd.to_datetime(calls.event_at,utc=True); calls['month']=calls.event_at.dt.to_period('M').astype(str)
C:\Users\LOQ\AppData\Local\Temp\ipykernel_9128\18749289.py:3: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  s=read('agent_sessions'); s['login_at']=pd.to_datetime(s.login_at,utc=True); s['logout_at']=pd.to_datetime(s.logout_at,utc=True); s['month']=s.l

,targeted_accounts,success_recovery,unique_payers,session_hours,recovery_per_target,payer_rate,recovery_per_agent_hour,answer_rate
2026-01,5732,1.872291e+08,2374,11162.5786,32663.8394,0.4142,16772.9280,0.2007
2026-02,5160,1.701425e+08,2173,10557.0114,32973.3438,0.4211,16116.5360,0.1978
2026-03,5666,1.889124e+08,2419,11130.8200,33341.4003,0.4269,16972.0087,0.1999
2026-04,5585,1.751380e+08,2304,10620.3217,31358.6470,0.4125,16490.8417,0.1933
2026-05,5800,1.842503e+08,2344,10833.1292,31767.2894,0.4041,17008.0386,0.2037
2026-06,5535,1.755597e+08,2286,10703.0086,31718.1079,0.4130,16402.8390,0.2040
2026-07,5666,1.872423e+08,2335,11180.1586,33046.6405,0.4121,16747.7289,0.1938


In [5]:
print('Jan → Jul recovery change:', round((m.success_recovery.iloc[-1]/m.success_recovery.iloc[0]-1)*100,2),'%')
print('Jan → Jul recovery/target change:', round((m.recovery_per_target.iloc[-1]/m.recovery_per_target.iloc[0]-1)*100,2),'%')
print('Jul vs Jun recovery:', round((m.success_recovery.iloc[-1]/m.success_recovery.iloc[-2]-1)*100,2),'%')
print('Jul vs Jun recovery/target:', round((m.recovery_per_target.iloc[-1]/m.recovery_per_target.iloc[-2]-1)*100,2),'%')
print('Jul vs Jun answer rate:', round((m.answer_rate.iloc[-1]/m.answer_rate.iloc[-2]-1)*100,2),'%')


Jan → Jul recovery change: 0.01 %
Jan → Jul recovery/target change: 1.17 %
Jul vs Jun recovery: 6.65 %
Jul vs Jun recovery/target: 4.19 %
Jul vs Jun answer rate: -5.0 %


### Interpretation
**Fact:** the 11% improvement is not sustained. Jan–Jul recovery is essentially flat and normalized recovery improves only ~1.17%. July raw recovery rises 6.65% versus June, but recovery per targeted account rises 4.19% and answer rate falls ~5%.

**Classification:** Fact. This does not establish why recovery moved.

## 3. Portfolio mix and drivers
The targeted portfolio is broadly stable across loan type, risk segment, timezone, schema version and DPD mix. Language and client are not present in the supplied schema, so those requested drivers cannot be tested.

In [6]:
for c in ['loan_type','risk_segment']:
    x=target.merge(accounts[['account_id',c]],on='account_id',how='left')
    share=x.groupby(['month',c]).account_id.nunique().groupby(level=0).transform(lambda s:s/s.sum())
    print(c); print((share.unstack().mul(100)).round(1))


loan_type
loan_type  AUTO  BNPL  CONSUMER  CREDIT_CARD  PERSONAL
month                                                 
2026-01    20.7  19.6      19.3         20.6      19.7
2026-02    20.6  19.7      19.6         19.8      20.3
2026-03    19.8  19.5      20.0         20.6      20.2
2026-04    19.8  19.9      20.0         20.3      20.0
2026-05    21.2  19.7      19.4         20.3      19.4
2026-06    19.8  19.6      19.6         20.5      20.5
2026-07    20.3  20.5      20.0         19.6      19.6
2026-08    20.1  19.5      20.6         20.8      19.1
risk_segment
risk_segment  HIGH   LOW  MEDIUM   NPA
month                                 
2026-01       25.0  25.6    24.7  24.7
2026-02       24.9  25.7    24.7  24.7
2026-03       24.9  25.2    26.0  23.9
2026-04       25.3  25.0    24.7  25.0
2026-05       24.5  25.3    25.3  24.8
2026-06       24.8  25.4    25.7  24.1
2026-07       25.1  25.1    25.1  24.7
2026-08       23.6  24.8    26.9  24.7


## 4. Channel association
We calculate a transparent 7-day post-touch association using qualifying channel events. Because channel exposure is observational, these are **correlations/associations**, not causal lift.

In [7]:
E=[]
E.append(calls[(calls.direction=='OUTBOUND')&(calls.call_status=='ANSWERED')][['account_id','event_at']].assign(channel='VOICE'))
sms=read('sms_events'); sms['event_at']=pd.to_datetime(sms.event_at,utc=True); E.append(sms[sms.event_type.isin(['DELIVERED','CLICKED'])][['account_id','event_at']].assign(channel='SMS'))
wa=read('whatsapp_events').drop_duplicates(); wa['event_at']=pd.to_datetime(wa.event_at,utc=True); E.append(wa[wa.event_type.isin(['DELIVERED','READ','REPLIED','PAYMENT_CLICK'])][['account_id','event_at']].assign(channel='WHATSAPP'))
f=read('field_visits'); f['event_at']=pd.to_datetime(f.event_at,utc=True); E.append(f[f.outcome.isin(['CONTACTED','PTP','PAID'])][['account_id','event_at']].assign(channel='FIELD'))
touches=pd.concat(E,ignore_index=True)
mat=touches.merge(p.query("payment_status=='SUCCESS'")[['payment_id','account_id','event_at','amount']],on='account_id',suffixes=('_touch','_pay'))
mat=mat[(mat.event_at_pay>=mat.event_at_touch)&(mat.event_at_pay<=mat.event_at_touch+pd.Timedelta(days=7))].sort_values(['payment_id','event_at_touch']).drop_duplicates('payment_id',keep='last')
base=touches.groupby('channel').account_id.nunique()
ch=mat.groupby('channel').agg(payers=('payment_id','nunique'),recovery=('amount','sum')).join(base.rename('touched_accounts'))
ch['payer_rate_7d']=ch.payers/ch.touched_accounts
ch['recovery_per_touched_account']=ch.recovery/ch.touched_accounts
ch.sort_values('payer_rate_7d',ascending=False).round(4)


,payers,recovery,touched_accounts,payer_rate_7d,recovery_per_touched_account
channel,,,,,
WHATSAPP,664,50370891.50,22026,0.0301,2286.8833
SMS,394,29202619.74,15689,0.0251,1861.3436
VOICE,305,23533011.29,12729,0.0240,1848.7714
FIELD,190,14404055.81,10237,0.0186,1407.0583


## 5. Counterfactual and ₹10 Cr decision
There is no clean treatment flag or documented strategy switch date in the supplied data. Therefore a causal estimate should not be fabricated.

**Recommendation:** Better borrower targeting, implemented as a staged experiment. Treatment should use a new targeting policy; control should retain current targeting. Primary KPI: incremental successful recovery per eligible account. Guardrails: complaints, PTP kept rate, and cost per ₹ recovered.

A full ROI/break-even figure is not identifiable from this dataset because channel/agent/telephony costs and randomized treatment/control data are absent.

## 6. Final answer
**The 11% claim is not proven.** The strongest defensible conclusion is that recovery is broadly flat after proper deduplication and normalization, while the data quality itself can inflate reporting. The next investment should be a controlled borrower-targeting pilot rather than an unconditional ₹10 Cr deployment.